# Analyzing word and document frequency: tf-idf

A central question in text mining and natural language processing is how to quantify what a document is about. Can we do this by looking at the words that make up the document?

One measure of how important a word may be is its term frequency (tf).

This is how frequently a word occurs in a document - as we saw in Lab 2. However, there are words in a document that occur many times but may not be important. In English these words are most often things like “the”, “is”, “of”, and so forth. We might take the approach of adding words like these to a list of stop words and removing them before analysis, but it is possible that some of these words might be more important in some documents than others. A list of stop words is not a very sophisticated approach to adjusting term frequency for commonly used words.

Another approach is to look at a term’s inverse document frequency (idf), which decreases the weight for commonly used words and increases the weight for words that are not used very much in a collection of documents. This can be combined with term frequency to calculate a term’s tf-idf (the two quantities multiplied together), the frequency of a term adjusted for how rarely it is used.

The tf-idf statistic is intended to measure how important a word is to a document in a collection (or corpus) of documents, for example, to one novel in a collection of novels or to one website in a collection of websites.

### Preparing data

In [1]:
import requests
import string
import pandas as pd

# Jane Eyre
book_url = 'https://www.gutenberg.org/files/1260/1260-0.txt'
response = requests.get(book_url)
bronte1 = response.text
allowed_chars = string.ascii_letters + string.digits + string.whitespace
bronte1 = ''.join(c for c in bronte1 if c in allowed_chars)

# Wuthering Heights
book_url = 'https://www.gutenberg.org/cache/epub/768/pg768.txt'
response = requests.get(book_url)
bronte2 = response.text
allowed_chars = string.ascii_letters + string.digits + string.whitespace
bronte2 = ''.join(c for c in bronte2 if c in allowed_chars)

# Vilette
book_url = 'https://www.gutenberg.org/files/9182/9182-0.txt'
response = requests.get(book_url)
bronte3 = response.text
allowed_chars = string.ascii_letters + string.digits + string.whitespace
bronte3 = ''.join(c for c in bronte3 if c in allowed_chars)

# Agnes Gray
book_url = 'https://www.gutenberg.org/files/767/767-0.txt'
response = requests.get(book_url)
bronte4 = response.text
allowed_chars = string.ascii_letters + string.digits + string.whitespace
bronte4 = ''.join(c for c in bronte4 if c in allowed_chars)

# Create our dataframes
bronte1_lines = bronte1.splitlines()

bronte1_df = pd.DataFrame({
    "line": bronte1_lines,
    "line_number": list(range(len(bronte1_lines)))
})

bronte2_lines = bronte2.splitlines()

bronte2_df = pd.DataFrame({
    "line": bronte2_lines,
    "line_number": list(range(len(bronte2_lines)))
})

bronte3_lines = bronte3.splitlines()

bronte3_df = pd.DataFrame({
    "line": bronte3_lines,
    "line_number": list(range(len(bronte3_lines)))
})

bronte4_lines = bronte4.splitlines()

bronte4_df = pd.DataFrame({
    "line": bronte4_lines,
    "line_number": list(range(len(bronte4_lines)))
})

# We’ll want to know which content comes from which book
bronte1_df = bronte1_df.assign(book = 'Jane Eyre')
bronte2_df = bronte2_df.assign(book = 'Wuthering Heights')
bronte3_df = bronte3_df.assign(book = 'Vilette')
bronte4_df = bronte4_df.assign(book = 'Agnes Grey')

# Finally, we concatenate the books into one dataframe
books = [bronte1_df, bronte2_df, bronte3_df, bronte4_df]
bronte_books_df = pd.concat(books)
bronte_books_df.head()

,line,line_number,book
0,START OF THE PROJECT GUTENBERG EBOOK 1260,0,Jane Eyre
1,,1,Jane Eyre
2,JANE EYRE,2,Jane Eyre
3,AN AUTOBIOGRAPHY,3,Jane Eyre
4,,4,Jane Eyre


In [2]:
# We split the data into words
# We first split the text column into a list of words
bronte_books_df['word'] = bronte_books_df['line'].str.split()

# Explode the words column to create a new row for each word (this creates a separate row for each word from the newly created words list)
bronte_books_df = bronte_books_df.explode('word')

# Reset the index of the dataframe (we want to index each word now)
bronte_books_df = bronte_books_df.reset_index(drop=True)
bronte_books_df.head()

,line,line_number,book,word
0,START OF THE PROJECT GUTENBERG EBOOK 1260,0,Jane Eyre,START
1,START OF THE PROJECT GUTENBERG EBOOK 1260,0,Jane Eyre,OF
2,START OF THE PROJECT GUTENBERG EBOOK 1260,0,Jane Eyre,THE
3,START OF THE PROJECT GUTENBERG EBOOK 1260,0,Jane Eyre,PROJECT
4,START OF THE PROJECT GUTENBERG EBOOK 1260,0,Jane Eyre,GUTENBERG


In [3]:
# For our investigations the line & line_number columns will not be necessary, so we will remove them
bronte_books_df = bronte_books_df[['book', 'word']]
bronte_books_df

,book,word
0,Jane Eyre,START
1,Jane Eyre,OF
2,Jane Eyre,THE
3,Jane Eyre,PROJECT
4,Jane Eyre,GUTENBERG
...,...,...
575971,Agnes Grey,THE
575972,Agnes Grey,PROJECT
575973,Agnes Grey,GUTENBERG
575974,Agnes Grey,EBOOK


### Word counting revisited

In [4]:
# Let's count the occurrences of each word - this is a prerequisite for finding term frequency
count_df = bronte_books_df.groupby('word')['word'].count() # Group by word column, then only keep the word column and perform the counting

# Let's sort by term frequency
count_df_sorted = count_df.sort_values(ascending=False)

count_df_sorted.head(10)

word
the    21916
and    19421
I      18440
to     15556
of     13036
a      12126
in      7929
was     7438
you     6263
her     5981
Name: word, dtype: int64

In [5]:
# The .size() method operates similary, but differs slightly in output format
# .size() also counts null values, which .count() does not
bronte_books_df.groupby(['word']).size().sort_values(ascending=False).reset_index(name='count')

,word,count
0,the,21916
1,and,19421
2,I,18440
3,to,15556
4,of,13036
...,...,...
30097,youyour,1
30098,youyoud,1
30099,youwiseralmost,1
30100,youwhich,1


In [6]:
# Groupby allows grouping based on multiple columns
bronte_books_df.groupby(['word', 'book']).size().sort_values(ascending=False).reset_index(name='count')

,word,book,count
0,the,Vilette,7725
1,the,Jane Eyre,7332
2,I,Jane Eyre,7009
3,and,Jane Eyre,6263
4,and,Vilette,6097
...,...,...,...
52935,18,Vilette,1
52936,1778,Wuthering Heights,1
52937,16,Jane Eyre,1
52938,1500,Wuthering Heights,1


### Aggregate
One useful and elegant way of counting/aggregating data in pandas is by using the .agg() method.


In [7]:
# We group our data by words, then we aggregate and can decide what information we want to display for each column

# setting 'first' for the book column means that in the new dataframe we will display the first book on which each word occurs (in the book column)
# setting 'count' for the word column means that in the new dataframe we will display the count of given word (in the word column)
count_df = bronte_books_df.groupby('word').agg({'book': 'first', 'word': 'count'})
count_df

# Another way to describe the line above is - for each group (in our case a group = a word and all its appearances) we show on the 'book' column the first book from that group and on the 'word' column the total count of entries from that group
# .agg() is more flexible than .apply() and allows multi-column aggregations like the one we see above, each of which can be different - e.g. first and count

,book,word
word,,
07042,Wuthering Heights,1
1,Wuthering Heights,4
10,Vilette,1
1260,Jane Eyre,2
13th,Jane Eyre,1
...,...,...
zigzag,Jane Eyre,2
zigzags,Vilette,1
zle,Vilette,1


In [8]:
# Because we used groupby, the 'word' keyword has become both an index and a column name
# To get rid of any naming problems down the line, we will rename the column name 'word' to 'count'
count_df = count_df.rename(columns={'word': 'count'})

# Sorting values based on count column
count_df.sort_values('count', ascending=False)

,book,count
word,,
the,Jane Eyre,21916
and,Jane Eyre,19421
I,Jane Eyre,18440
to,Jane Eyre,15556
of,Jane Eyre,13036
...,...,...
youyour,Jane Eyre,1
youyoud,Jane Eyre,1
youwiseralmost,Jane Eyre,1


### Merging Dataframes

What we want next is to have a dataframe in which we know how many times each word appears per book and how many times it appears in all of the books.

It is sometimes very useful to merge together two dataframes and this is what we're going to do to get our desired dataframe.

In [9]:
count_df_1 = bronte_books_df.groupby(['word', 'book']).size().sort_values(ascending=False).reset_index(name='count') # How many appearances each word has in each book
count_df_1

,word,book,count
0,the,Vilette,7725
1,the,Jane Eyre,7332
2,I,Jane Eyre,7009
3,and,Jane Eyre,6263
4,and,Vilette,6097
...,...,...,...
52935,18,Vilette,1
52936,1778,Wuthering Heights,1
52937,16,Jane Eyre,1
52938,1500,Wuthering Heights,1


In [10]:
count_df_2 = bronte_books_df.groupby(['book']).size().sort_values(ascending=False).reset_index(name='count') # How many words each book has
count_df_2

,book,count
0,Vilette,196246
1,Jane Eyre,189694
2,Wuthering Heights,121120
3,Agnes Grey,68916


In [11]:
book_words = count_df_1.merge(count_df_2, on='book')
book_words.head(10)

,word,book,count_x,count_y
0,the,Vilette,7725,196246
1,the,Jane Eyre,7332,189694
2,I,Jane Eyre,7009,189694
3,and,Jane Eyre,6263,189694
4,and,Vilette,6097,196246
5,I,Vilette,5762,196246
6,to,Jane Eyre,5030,189694
7,of,Vilette,4813,196246
8,to,Vilette,4656,196246
9,and,Wuthering Heights,4502,121120


In [12]:
book_words = book_words.rename(columns={'count_x': 'word_appearances_in_book', 'count_y': 'book_total_word_count'}) # Give more meaningful names
book_words.head(10)

,word,book,word_appearances_in_book,book_total_word_count
0,the,Vilette,7725,196246
1,the,Jane Eyre,7332,189694
2,I,Jane Eyre,7009,189694
3,and,Jane Eyre,6263,189694
4,and,Vilette,6097,196246
5,I,Vilette,5762,196246
6,to,Jane Eyre,5030,189694
7,of,Vilette,4813,196246
8,to,Vilette,4656,196246
9,and,Wuthering Heights,4502,121120


### Exercise 1

1. Add a **tf** (term frequency) column to your book_words dataframe.
2. Add a new idf column to your dataframe
3. Add the final tf-idf column to your dataframe
4. Display your dataframe's words in descending order of their tf-idf.

Term frequency says how frequently a given word appears in a book. The formula for calculating it is
    
    term_frequency = word_appearances_in_book / book_total_word_count

Idf or inverse document frequency is computed as **idf = log(N / n)**

where


```
N is the total number of documents (books) in your dataset and n is the number of documents containing the word.
```



Once you have tf and idf, the tf-idf is obtained by simply multiplying the two.

Hint: For ex. 1.2 the pandas **transform** function could come in handy.

In [13]:
import numpy as np

# 1) Term frequency (tf)
book_words['tf'] = book_words['word_appearances_in_book'] / book_words['book_total_word_count']

# 2) Inverse document frequency (idf = log(N / n))
N = book_words['book'].nunique()
book_words['n_docs_with_word'] = book_words.groupby('word')['book'].transform('nunique')
book_words['idf'] = np.log(N / book_words['n_docs_with_word'])

# 3) tf-idf
book_words['tf_idf'] = book_words['tf'] * book_words['idf']

# 4) Display words in descending order of tf-idf
book_words.sort_values('tf_idf', ascending=False)[['word', 'book', 'tf', 'idf', 'tf_idf']].head(20)

,word,book,tf,idf,tf_idf
158,Heathcliff,Wuthering Heights,0.003410,1.386294,0.004727
200,Linton,Wuthering Heights,0.002807,1.386294,0.003892
211,Rochester,Jane Eyre,0.001645,1.386294,0.002280
203,Catherine,Wuthering Heights,0.002749,0.693147,0.001906
409,Hareton,Wuthering Heights,0.001354,1.386294,0.001877
802,Murray,Agnes Grey,0.001190,1.386294,0.001649
299,Dr,Vilette,0.001147,1.386294,0.001589
311,Bretton,Vilette,0.001101,1.386294,0.001526
334,Graham,Vilette,0.001009,1.386294,0.001399
955,Weston,Agnes Grey,0.001001,1.386294,0.001388


# Language Models

A language model is a statistical model that can be used to estimate the probability of a sequence of words in a language. It is trained on a corpus of text data, and learns to predict the likelihood of observing a given sequence of words based on the frequency and context of those words in the training data.

Language models can be used for a variety of natural language processing tasks, such as text generation, machine translation, speech recognition, and more.

In [14]:
import nltk
from nltk.corpus import brown
from nltk import FreqDist
nltk.download('brown')

# load the Brown corpus
corpus = brown.words()

[nltk_data] Downloading package brown to
[nltk_data]     C:\Users\alexc\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\brown.zip.


In this example, we're using the Brown corpus from the nltk library, which is a collection of text samples from a wide range of genres, including news, fiction, and academic writing.

In [15]:
print(corpus[1100:1110]) # Print a sample of 10 words from the corpus

['voters', '.', 'Despite', 'the', 'warning', ',', 'there', 'was', 'a', 'unanimous']


In [21]:
# create a frequency distribution of the words in the corpus
freq_dist = FreqDist(corpus)

# calculate the total number of words in the corpus
total_words = len(corpus)

# calculate the probability of each word in the corpus
word_probs = {word: freq_dist[word] / total_words for word in freq_dist.keys()}
print(word_probs['high']) # Probability of the word 'high' to appear

0.0003970058353829513


### Naive sentence generation

We're going to create a naive function that generates sentences using our language model.

In [33]:
# generate a sentence using the language model
import random

def generate_sentence(word_length = 10):
    sentence = []
    while len(sentence) < word_length:
        word = random.choices(list(word_probs.keys()), list(word_probs.values()))[0]
        sentence.append(word)
    return " ".join(sentence)

In [34]:
print(generate_sentence())

nutrition place down discounts know wished , qualifications outside 1


The sentences generated are likely not going to sound very good, since the model is extremely naive.

All that is happening is that each word in the sentence gets semi-randomly generated with the likelihood of it being chosen depending on its frequency in the Brown corpus.

# N-grams

So far we’ve considered words as individual units, and considered the relationship to their frequency of occurrence. However, many interesting text analyses are based on the relationships between words.
One such relationship is given by n-grams.

N-grams are groups of n consecutive words that appear in a given text corpus.

Bigrams are groups of 2 consecutive words (e.g. she went, he ate, car crashed)

Trigrams are groups of 3 consecutive words (e.g. she went home, he ate a, the car crashed).

In [35]:
# Example of what bigrams look like
bigrams = list(nltk.bigrams(corpus))
bigrams[:10]

[('The', 'Fulton'),
 ('Fulton', 'County'),
 ('County', 'Grand'),
 ('Grand', 'Jury'),
 ('Jury', 'said'),
 ('said', 'Friday'),
 ('Friday', 'an'),
 ('an', 'investigation'),
 ('investigation', 'of'),
 ('of', "Atlanta's")]

In [38]:
# Example of what trigrams look like
trigrams = list(nltk.trigrams(corpus))
trigrams[5:20]

[('said', 'Friday', 'an'),
 ('Friday', 'an', 'investigation'),
 ('an', 'investigation', 'of'),
 ('investigation', 'of', "Atlanta's"),
 ('of', "Atlanta's", 'recent'),
 ("Atlanta's", 'recent', 'primary'),
 ('recent', 'primary', 'election'),
 ('primary', 'election', 'produced'),
 ('election', 'produced', '``'),
 ('produced', '``', 'no'),
 ('``', 'no', 'evidence'),
 ('no', 'evidence', "''"),
 ('evidence', "''", 'that'),
 ("''", 'that', 'any'),
 ('that', 'any', 'irregularities')]

### Naive next word prediction

Knowing that word relations are pretty important in our language, let's create a function that predicts what the next word in a sentence would be using a simple **bigram** language model.

In [39]:
from nltk.corpus import brown
import random

# get the words from the Brown corpus
corpus = brown.words()

# create bigrams from the corpus
bigrams = list(nltk.bigrams(corpus))

# calculate the frequency distribution of the bigrams
bigram_freqdist = nltk.FreqDist(bigrams)

# calculate the total number of bigrams in the corpus
total_bigrams = len(bigrams)

# create a function to generate the next word based on the previous word
def generate_next_word(sentence):
    prev_word = sentence.split()[-1]
    possible_words = {}
    for bigram in bigram_freqdist:
        if bigram[0] == prev_word:
            possible_words[bigram[1]] = bigram_freqdist[bigram] / total_bigrams
    if possible_words:
        return max(possible_words, key=possible_words.get)
    else:
        return None

In [40]:
# predict the next word for a given context
context = "The director"
next_word = generate_next_word(context)
print(f"The predicted next word for '{context}' is '{next_word}'")

The predicted next word for 'The director' is 'of'


### Exercise 2
1. Create a function that takes as input the number of words to generate and generates a sentence using the previous bigram language model. You can start with a random first word from the brown corpus and then use generate_next_word(sentence) function to help you.

2. Create a function that predicts the next word of a sentence by looking at the previous two words. This means you will create a trigram language model - use the same Brown corpus as before.

In [43]:
import random
import nltk

def generate_sentence_bigram(num_words):
    if num_words <= 0:
        return ""

    sentence_words = [random.choice(list(corpus))]  # random first word

    while len(sentence_words) < num_words:
        current_sentence = " ".join(sentence_words)
        next_word = generate_next_word(current_sentence) 

    

        sentence_words.append(next_word)

    return " ".join(sentence_words)


print("Bigram-generated sentence:")
print(generate_sentence_bigram(12))

trigram_freqdist = nltk.FreqDist(nltk.trigrams(corpus))

def predict_next_word_trigram(sentence):
    words = sentence.split()
    if len(words) < 2:
        return None

    prev_word1, prev_word2 = words[-2], words[-1]
    candidates = {}

    for (w1, w2, w3), count in trigram_freqdist.items():
        if w1 == prev_word1 and w2 == prev_word2:
            candidates[w3] = count

    if candidates:
        return max(candidates, key=candidates.get)

    return None


context = "The female"
print(f"Trigram next-word prediction for '{context}': {predict_next_word_trigram(context)}")

Bigram-generated sentence:
by the same time , and the same time , and the
Trigram next-word prediction for 'The female': parasite


### N-grams in dataframes

Let's get back to our books.
We'll create a dataframe containing information about the bigrams in our books corpus.


In [44]:
# The simplest way to do this would be to create the dataframe directly from bigrams rather than unigrams (single words)
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

bronte1_bigrams = list(nltk.bigrams(nltk.word_tokenize(bronte1)))
bronte1_df = pd.DataFrame(bronte1_bigrams, columns=['Word 1', 'Word 2'])

bronte2_bigrams = list(nltk.bigrams(nltk.word_tokenize(bronte2)))
bronte2_df = pd.DataFrame(bronte2_bigrams, columns=['Word 1', 'Word 2'])

bronte3_bigrams = list(nltk.bigrams(nltk.word_tokenize(bronte3)))
bronte3_df = pd.DataFrame(bronte3_bigrams, columns=['Word 1', 'Word 2'])

bronte4_bigrams = list(nltk.bigrams(nltk.word_tokenize(bronte4)))
bronte4_df = pd.DataFrame(bronte4_bigrams, columns=['Word 1', 'Word 2'])


# We’ll want to know which content comes from which book
bronte1_df = bronte1_df.assign(book = 'Jane Eyre')
bronte2_df = bronte2_df.assign(book = 'Wuthering Heights')
bronte3_df = bronte3_df.assign(book = 'Vilette')
bronte4_df = bronte4_df.assign(book = 'Agnes Grey')

# Finally, we concatenate the books into one dataframe
books = [bronte1_df, bronte2_df, bronte3_df, bronte4_df]
bronte_books_df = pd.concat(books)
bronte_books_df.head()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\alexc\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\alexc\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


,Word 1,Word 2,book
0,START,OF,Jane Eyre
1,OF,THE,Jane Eyre
2,THE,PROJECT,Jane Eyre
3,PROJECT,GUTENBERG,Jane Eyre
4,GUTENBERG,EBOOK,Jane Eyre


### Exercise 3

1. Add a **bigram** column that shows the entire bigrams ("The Project" and "Project Gutenberg" are examples of this column's values), not just the separate words.
2. Clean the dataframe by removing stop words.
3. Display the most frequently occuring 10 bigrams.

In [42]:
# Write your code below


### Exercise 4

1. Create a dataframe containing the **bigram, word1, word2** and **book** columns for the following 4 books and remove stop words:
        https://www.gutenberg.org/cache/epub/1228/pg1228.txt - On the Origin of Species, by Charles Darwin

        https://www.gutenberg.org/cache/epub/4363/pg4363.txt - Beyond Good and Evil, by Friedrich Nietzsche

        https://www.gutenberg.org/cache/epub/3296/pg3296.txt - The Confessions of Saint Augustine, by Saint Augustine

        https://www.gutenberg.org/files/1661/1661-0.txt - The Adventures of Sherlock Holmes, by Arthur Conan Doyle

2. Display the most frequent 8 words of each book (use word1 column when counting)

3. Display the most relevant 8 words of each book based on tf-idf (use word1 column when counting)

4. Display the most relevant 5 bigrams of each book based on tf-idf

5. Display the most frequent 5 street names found in the entire 4 book corpus. The book they are coming from should also be visible.

6. Choose a fixed word1 of your choice and find the most common 5 bigrams in each book that have word1 equal to the word you chose.




In [ ]:
# Write your code below


# Exercise 5
1. Create a dataframe containing all the columns you will need to do a trigram-based analysis for **a book of your choice**. Suggested columns: trigram, word1, word2, word3, book and potentially tf, idf, tf-idf.

2. Display the top 10 words by tf-idf
3. Display the most frequent 5 trigrams for 2 target words of your choice.

 This means you will choose 2 separate words from the book (suggestion is to choose relevant words, e.g. main character names), keep one of the 3 words from the trigram fixed, and then display the most frequent trigrams co-occurring with your word. E.g. You choose 'John' as one of your two words - then you set it as a fixed word (up to you if it should be word1, word2 or word3), and find what are the most frequent trigrams that have John in that fixed position.

 Do this for two separate words, some suggestions are to use the protagonist and the antagonist of your story, or to use opposing principles/subjects from your work as your target words.




In [43]:
# Write your code below
